In [11]:
import json
from pathlib import Path
import glob, os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
#DEVICE = "cpu"

# your basic TCN settings
K = 25            # max number of boxes per frame
#NUM_FEATURES = 30 # number of features for every person(ex. boxes and keypoints)
NUM_FEATURES = 28 + 12 # 12xy 4bbx + joint_mask for 12 keypoints
WINDOW_SIZE = 16 # ~3 seconds if ~5 jsons/sec
WINDOW_STEP = 4  # move by 1 step
NUM_CLASSES = 1#3 # 0=none, 1=tension, 2=fight
DS_PATH = '/home/anna/TCN_TRAIN/data/dataset_v0/jsons/'
#B = 32

Device: cpu


In [12]:
def get_subfolders_files(path):
    """
    Returns a list of all subfolders within the given path.
    """
    p = Path(path)
    #print(p)
    subfolders = [os.path.join(path, entry.name) for entry in p.iterdir() if entry.is_dir()]
    jsons = [os.path.join(folder_path, entry)  for folder_path in subfolders for entry in os.listdir(folder_path)]
    #print(jsons)
    return jsons

#json_paths = glob.glob('/home/anna/TCN_TRAIN/DS_tension_fight_tcn/jsons_corrected/*.json')
json_paths = get_subfolders_files(DS_PATH)
print("Found jsons:", len(json_paths))

with open(json_paths[0], "r") as f:
    data_example = json.load(f)

data_example.keys()


Found jsons: 87


dict_keys(['video', 'fps', 'step', 'frames'])

In [4]:
from pathlib import Path
from sklearn.model_selection import train_test_split

def list_jsons(folder):
    return sorted(str(p) for p in Path(folder).glob("*.json"))

def split_folder(paths, test_size=0.2, seed=42):
    if len(paths) < 2:
        # too small to split safely
        return paths, []
    return train_test_split(paths, test_size=test_size, random_state=seed)
    
def list_files_pathlib(directory_path, extension='*.json'):
    """
    Recursively finds all files with a given extension using pathlib.
    
    Args:
        directory_path (str): The starting directory to search.
        extension (str): The file extension (e.g., '*.txt', '*.jpg').
        
    Returns:
        list: A list of Path objects for the matching files.
    """
    p = Path(directory_path)
    # Use rglob (recursive glob) with the extension pattern
    files = list(p.rglob(extension))
    return files
    
normal1 = "/home/anna/TCN_TRAIN/DS_tension_fight_tcn/"
normal2 = "/home/anna/TCN_TRAIN/DS_tension_fight_tcn/"
events  = "/home/anna/TCN_TRAIN/DS_tension_fight_tcn/"
folders = [os.path.join(DS_PATH, entry.name) for entry in Path(DS_PATH).iterdir() if entry.is_dir()]
print(folders)
train_paths, val_paths = [], []

for folder in folders:# [normal1, normal2, events]:
    paths = list_jsons(folder)
    
    tr, va = split_folder(paths, test_size=0.2, seed=42)
    train_paths += tr
    val_paths   += va

# optional: save for reproducibility
Path("splits").mkdir(exist_ok=True)
Path("splits/train_paths.txt").write_text("\n".join(train_paths))
Path("splits/val_paths.txt").write_text("\n".join(val_paths))


['/home/anna/TCN_TRAIN/data/processed/dataset_v0/jsons/usual_jsons_from_events', '/home/anna/TCN_TRAIN/data/processed/dataset_v0/jsons/usual_jsons_from_cams', '/home/anna/TCN_TRAIN/data/processed/dataset_v0/jsons/jsons_corrected']


1663

In [7]:
import yaml

In [13]:
#files = 
list_files_pathlib(DS_PATH)

[PosixPath('/home/anna/TCN_TRAIN/data/dataset_v0/jsons/usual_jsons_from_events/event_27.json'),
 PosixPath('/home/anna/TCN_TRAIN/data/dataset_v0/jsons/usual_jsons_from_events/event_30.json'),
 PosixPath('/home/anna/TCN_TRAIN/data/dataset_v0/jsons/usual_jsons_from_events/event_28.json'),
 PosixPath('/home/anna/TCN_TRAIN/data/dataset_v0/jsons/usual_jsons_from_events/event_12.json'),
 PosixPath('/home/anna/TCN_TRAIN/data/dataset_v0/jsons/usual_jsons_from_events/event_11.json'),
 PosixPath('/home/anna/TCN_TRAIN/data/dataset_v0/jsons/usual_jsons_from_events/event_7.json'),
 PosixPath('/home/anna/TCN_TRAIN/data/dataset_v0/jsons/usual_jsons_from_events/event_19.json'),
 PosixPath('/home/anna/TCN_TRAIN/data/dataset_v0/jsons/usual_jsons_from_events/event_1.json'),
 PosixPath('/home/anna/TCN_TRAIN/data/dataset_v0/jsons/usual_jsons_from_events/event_6.json'),
 PosixPath('/home/anna/TCN_TRAIN/data/dataset_v0/jsons/usual_jsons_from_events/event_10.json'),
 PosixPath('/home/anna/TCN_TRAIN/data/datas

In [8]:
train_paths

['/home/anna/TCN_TRAIN/data/processed/dataset_v0/jsons/usual_jsons_from_events/event_3.json',
 '/home/anna/TCN_TRAIN/data/processed/dataset_v0/jsons/usual_jsons_from_events/event_1.json',
 '/home/anna/TCN_TRAIN/data/processed/dataset_v0/jsons/usual_jsons_from_events/event_7.json',
 '/home/anna/TCN_TRAIN/data/processed/dataset_v0/jsons/usual_jsons_from_events/event_21.json',
 '/home/anna/TCN_TRAIN/data/processed/dataset_v0/jsons/usual_jsons_from_events/event_24.json',
 '/home/anna/TCN_TRAIN/data/processed/dataset_v0/jsons/usual_jsons_from_events/event_2.json',
 '/home/anna/TCN_TRAIN/data/processed/dataset_v0/jsons/usual_jsons_from_events/event_26.json',
 '/home/anna/TCN_TRAIN/data/processed/dataset_v0/jsons/usual_jsons_from_events/event_10.json',
 '/home/anna/TCN_TRAIN/data/processed/dataset_v0/jsons/usual_jsons_from_events/event_13.json',
 '/home/anna/TCN_TRAIN/data/processed/dataset_v0/jsons/usual_jsons_from_events/event_14.json',
 '/home/anna/TCN_TRAIN/data/processed/dataset_v0/jsons

In [9]:
val_paths

['/home/anna/TCN_TRAIN/data/processed/dataset_v0/jsons/usual_jsons_from_events/event_8.json',
 '/home/anna/TCN_TRAIN/data/processed/dataset_v0/jsons/usual_jsons_from_events/event_25.json',
 '/home/anna/TCN_TRAIN/data/processed/dataset_v0/jsons/usual_jsons_from_events/event_20.json',
 '/home/anna/TCN_TRAIN/data/processed/dataset_v0/jsons/usual_jsons_from_events/event_30.json',
 '/home/anna/TCN_TRAIN/data/processed/dataset_v0/jsons/usual_jsons_from_events/event_17.json',
 '/home/anna/TCN_TRAIN/data/processed/dataset_v0/jsons/usual_jsons_from_events/event_18.json',
 '/home/anna/TCN_TRAIN/data/processed/dataset_v0/jsons/usual_jsons_from_cams/cam1001.json',
 '/home/anna/TCN_TRAIN/data/processed/dataset_v0/jsons/usual_jsons_from_cams/cam2801.json',
 '/home/anna/TCN_TRAIN/data/processed/dataset_v0/jsons/usual_jsons_from_cams/cam2601.json',
 '/home/anna/TCN_TRAIN/data/processed/dataset_v0/jsons/usual_jsons_from_cams/cam1101.json',
 '/home/anna/TCN_TRAIN/data/processed/dataset_v0/jsons/usual_js

In [10]:
 # this depends on your format, I'm assuming something like:
# data["frames"] = [ { "detections": [...], "events": [...] }, ... ]

frame0 = data_example["frames"][0]
frame0.keys()

dict_keys(['f', 't', 'individual_events', 'group_events', 'bbs_list_of_keypoints'])

In [11]:
len(frame0['bbs_list_of_keypoints'][6][6])

26

In [12]:
def norm_kps_xy_flat(kps_xy_flat, cx, cy, w, h, eps=1e-6):
    # kps_xy_flat: list/tuple length 2*V
    inv_w = 1.0 / (w + eps)
    inv_h = 1.0 / (h + eps)

    out = []
    for i, val in enumerate(kps_xy_flat):
        if i % 2 == 0:  # x
            out.append((val - cx) * inv_w)
        else:           # y
            out.append((val - cy) * inv_h)
    return out

def norm_kps_xy_flat_with_mask(kps_xy_flat, cx, cy, w, h, eps=1e-6):
    inv_w = 1.0 / (w + eps)
    inv_h = 1.0 / (h + eps)

    kps_norm = []
    joint_mask = []

    for j in range(0, len(kps_xy_flat), 2):
        x = kps_xy_flat[j]
        y = kps_xy_flat[j+1]

        visible = not (x == 0 and y == 0)
        #joint_mask.append(1 if visible else 0)

        #if visible:
        kps_norm.extend([(x - cx) * inv_w, (y - cy) * inv_h, 1.00 if visible else 0.00])
        #else:
            # keep zeros for missing (but mask tells the truth)
            #kps_norm.extend([0.0, 0.0])

    return kps_norm#, joint_mask

def frame_to_vector(frame, K):
    """
    frame: dict for one time step
    K: max number of boxes per frame
    """
    detections = frame['bbs_list_of_keypoints']  # <-- adapt this key name

    # sort boxes by area desc
    dets_sorted = sorted(
        detections,
        key=lambda d: (d[4] - d[2]) * (d[5] - d[3]),
        reverse=True,
    )

    boxes = []
    keypoints = []
    for d in dets_sorted[:K]:
        x1, y1, x2, y2 = d[2:6]
        w = x2 - x1
        h = y2 - y1
        cx = (x1 + x2 )/ 2
        cy = (y1 + y2 )/ 2

        # normalize
        '''cx /= img_w
        cy /= img_h
        w  /= img_w
        h  /= img_h'''
        kps = d[6][2:]
        # choose the correct normalizer for your keypoint format:
        kps_norm = norm_kps_xy_flat_with_mask(kps, cx, cy, w, h)          # xy only + mask
        #print(len(kps_norm))
        # kps_norm = norm_kps_xyc_flat(kps, cx, cy, w, h)       # if x,y,conf triplets
        boxes.extend([cx, cy, w, h]+ kps_norm)

    # pad if fewer than K boxes
    if len(dets_sorted) < K:
        boxes.extend([0.0] * NUM_FEATURES * (K - len(dets_sorted)))

    n_people = len(detections)
    n_people_norm = len(detections)/25

    x_t = boxes + [float(n_people_norm)]
    return np.array(x_t, dtype=np.float32)
    
def events_to_label(frame):
    # adjust key to your json
    events = frame.get("group_events", [])  

    if 4 in events:
        #return 2
        return 1
    if 3 in events:
        return 1
    return 0

In [13]:
#NOT TO USE
def add_ramp_before_after_events(
    y,
    ramp=(0.125, 0.25, 0.375, 0.5, 0.625, 0.75, 0.875),
):
    """
    y: 1D numpy array of 0/1 labels (per frame) for your joined class (danger)
    ramp: values to assign to frames around each event segment.
          The value closest to the event is ramp[-1] (largest), farthest is ramp[0].
          This function does NOT change frames where orig==1 (true event frames).
    """
    y = np.asarray(y, dtype=float).copy()
    orig = (y >= 0.5).astype(np.uint8)  # safe even if y already has floats
    L = len(y)

    # event starts: index s where orig[s]=1 and previous is 0 (or s==0)
    prev = np.r_[0, orig[:-1]]
    starts = np.flatnonzero((orig == 1) & (prev == 0))

    # event ends: index e where orig[e]=1 and next is 0 (or e==L-1)
    nxt = np.r_[orig[1:], 0]
    ends = np.flatnonzero((orig == 1) & (nxt == 0))

    ramp = np.asarray(ramp, dtype=float)
    K = len(ramp)

    for j in range(1, K + 1):
        val = ramp[-j]  # closest frame gets the largest value

        # before starts
        idx = starts - j
        m = (idx >= 0)
        idx = idx[m]
        # only paint where there is no true event (prevents overwriting)
        idx = idx[orig[idx] == 0]
        y[idx] = np.maximum(y[idx], val)

        # after ends (first after is end+1)
        idx = ends + j
        m = (idx < L)
        idx = idx[m]
        idx = idx[orig[idx] == 0]  # prevents overwriting next event if too close
        y[idx] = np.maximum(y[idx], val)

    return np.clip(y, 0.0, 1.0)


In [14]:
def json_to_sequence(path, K):
    with open(path, "r") as f:
        data = json.load(f)

    frames = data["frames"]  # adapt if needed

    features = []
    labels = []

    for frame in frames:
        x_t = frame_to_vector(frame, K)
        y_t = events_to_label(frame)

        features.append(x_t)
        labels.append(y_t)

    features = np.stack(features).astype(np.float32)  # (N, C)
    labels   = np.array(labels, dtype=np.float32)       # (N,)
    #labels_ramped = add_ramp_before_after_events(labels)
    return features, labels#_ramped

#X_seq, y_seq = json_to_sequence(path, K)
#assert X_seq.shape[1] == NUM_FEATURES * K + 1, (X_seq.shape, NUM_FEATURES, K)


In [15]:
len(json_paths)
print(len(train_paths))

68


In [16]:
print(train_paths[-2])
X_seq, y_seq = json_to_sequence(train_paths[-1], K)
X_seq.shape, y_seq.shape, y_seq

/home/anna/TCN_TRAIN/data/processed/dataset_v0/jsons/jsons_corrected/cam6_4.json


((532, 1001),
 (532,),
 array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1

In [17]:
class EventJsonDataset(Dataset):
    def __init__(self, json_paths, K,
                 window_size=16, window_step=4):
        self.samples = []
        self.window_size = window_size
        self.window_step = window_step
        self.K = K
        #self.img_w = img_w
        #self.img_h = img_h

        for path in json_paths:
            X_seq, y_seq = json_to_sequence(path, K)
            N = len(X_seq)
            if N < window_size:
                continue

            for start in range(0, N - window_size + 1, window_step):
                end = start + window_size
                X_win = X_seq[start:end] # (T, C)
                y_win = y_seq[start:end]    # (T,)
                #ignore examples of dataset where events are at the very beginning and at the very end
                count_pos_beg = np.sum(y_win[:window_size//2] == 1)
                count_pos_end = np.sum(y_win[window_size//2:] == 1)
                if y_win[window_size//2] == 1 or (y_win[window_size//2] == 0 and (count_pos_beg == 0 and count_pos_end == 0)):
                    y_central = y_win[window_size//2]
                
                    # if window has only 'none', keep only a fraction
                    #if np.all(y_win == 0):
                        #if np.random.rand() > 0.2:  # keep ~20% of pure-none windows
                            #continue
    
                    self.samples.append((X_win, y_central))
                else:
                    continue
                
        print("Total windows:", len(self.samples))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        X_win, y_last = self.samples[idx]
        # convert to tensors here
        X_win = torch.from_numpy(X_win)        # (T, C)
        y_last = torch.tensor(y_last)        # (1,)
        return X_win, y_last


In [18]:
dataset = EventJsonDataset(val_paths, K,
                           window_size=WINDOW_SIZE,
                           window_step=WINDOW_STEP)
X_win, y_last = dataset[101]
X_win.shape, y_last

Total windows: 4225


(torch.Size([16, 1001]), tensor(0.))

In [19]:
X_win, y_last = dataset[10]
X_win.shape, y_last

(torch.Size([16, 1001]), tensor(0.))

In [20]:
len(dataset)

4225

In [21]:
import torch
import torch.nn as nn

class BoneMLPEncoder(nn.Module):
    def __init__(self, K=25, V=12, edges=[(0,1),(0,2),(2,4),(1,3),(3,5),(0,6),(1,7),(6,7),(6,8),(8,10),(7,9),(9,11)],
                 out_dim=32, hidden=128, use_conf=True, add_bbox=True,
                 add_masks=True, conf_thr=0.2, min_visible_joints=3):
        super().__init__()
        self.K = K
        self.V = V
        self.edges = edges
        self.use_conf = use_conf
        self.add_bbox = add_bbox
        self.add_masks = add_masks
        self.conf_thr = conf_thr
        self.min_visible_joints = min_visible_joints

        nbones = len(edges)

        in_dim = 2 #anchor centroid
        #in_dim = (2 * V)                 # xy
        #if use_conf:
            #in_dim += (1 * V)            # conf
        in_dim += (2 * nbones)           # bone dxdy
        if add_bbox:
            in_dim += 4

        # add explicit masks as features (recommended)
        if add_masks:
            #in_dim += (1 * V)            # joint_mask
            in_dim += (1 * nbones)       # bone_mask
            in_dim += 1+1                  # person_mask anchor_validate

        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, out_dim),
            nn.ReLU(),
        )

    def forward(self, pose_xyc, bbox_cxcywh=None):
        """
        pose_xyc: (B,T,K,V,3) preferred (x,y,conf). If (B,T,K,V,2), conf is unavailable.
        bbox_cxcywh: (B,T,K,4) where padded persons have w=h=0 (recommended).
        Returns:
            emb: (B,T,K,out_dim)
            person_mask: (B,T,K) bool
        """
        num_persons = pose_xyc[..., -1]
        persons = pose_xyc[..., :-1].contiguous().view(*pose_xyc.shape[:-1], self.K, -1)
        #print(persons.shape)
        pers_keypoints = persons[..., 4:].contiguous().view(*persons.shape[:-1], self.V, -1)
        #print(pers_keypoints.shape)
        pers_bboxes = persons[..., :4]
        #print(pers_bboxes.shape)
        xy = pers_keypoints[..., :2]  # (B,T,K,V,2)
        #print(xy.shape)
        # --- joint mask ---
        '''if pose_xyc.size(-1) >= 3:
            conf = pose_xyc[..., 2]  # (B,T,K,V)
            joint_mask = conf > self.conf_thr
        else:
            conf = None
            # WARNING: only valid if (0,0) truly indicates missing joints
            joint_mask = (xy.abs().sum(dim=-1) > 0)  # (B,T,K,V)
    
        # --- person mask ---
        if bbox_cxcywh is not None:
            w = bbox_cxcywh[..., 2]  # (B,T,K)
            h = bbox_cxcywh[..., 3]  # (B,T,K)
            person_mask = (w > 0) & (h > 0)
        else:
            person_mask = joint_mask.any(dim=-1)  # (B,T,K)'''
    
        if pers_keypoints.size(-1) >= 3:
            conf = pers_keypoints[..., 2]  # (B,T,K,V)
            joint_mask = conf > self.conf_thr
        else:
            conf = None
            # WARNING: only valid if (0,0) truly indicates missing joints
            joint_mask = (xy.abs().sum(dim=-1) > 0)  # (B,T,K,V)
    
        # --- person mask ---
        '''if bbox_cxcywh is not None:
            w = bbox_cxcywh[..., 2]  # (B,T,K)
            h = bbox_cxcywh[..., 3]  # (B,T,K)
            person_mask = (w > 0) & (h > 0)
        else:
            person_mask = joint_mask.any(dim=-1)  # (B,T,K)'''
    
        w = pers_bboxes[..., 2]  # (B,T,K)
        h = pers_bboxes[..., 3]  # (B,T,K)
        person_mask = (w > 0) & (h > 0)
    
        #person_mask = joint_mask.any(dim=-1)  # (B,T,K)
    
        # require enough visible joints (optional but usually helpful)
        person_mask = person_mask & (joint_mask.sum(dim=-1) >= self.min_visible_joints)
    
        # if person invalid, force all its joints invalid
        joint_mask = joint_mask & person_mask.unsqueeze(-1)  # (B,T,K,V)
    
        # --- bones + bone_mask ---
        bones = []
        bone_masks = []
        for (u, v) in self.edges:
            bm = joint_mask[..., u] & joint_mask[..., v]                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                # (B,T,K)
            dv = xy[..., v, :] - xy[..., u, :]                    # (B,T,K,2)
    
            #if self.use_conf and conf is not None:
                #wgt = torch.sqrt(torch.clamp(conf[..., u] * conf[..., v], 0.0, 1.0))  # (B,T,K)
                #dv = dv * wgt.unsqueeze(-1)
    
            dv = dv * bm.unsqueeze(-1).float()                    # gate invalid bones to zero
            bones.append(dv)
            bone_masks.append(bm)
    
        bones = torch.stack(bones, dim=-2)                         # (B,T,K,Ebones,2)
        bone_mask = torch.stack(bone_masks, dim=-1)                # (B,T,K,Ebones)
    
        both_hips = joint_mask[...,6] & joint_mask[...,7]     # (B,T,K)
        midhip = 0.5*(xy[...,6,:] + xy[...,7,:])              # (B,T,K,2)
        
        anchor_xy = torch.where(both_hips[..., None], midhip, torch.zeros_like(midhip))
        anchor_valid = both_hips
    
        # --- anchor: mid-hip if possible else centroid of visible joints ---
        # hips in your reindexed 12-joint set: left_hip=6, right_hip=7
        '''lh_ok = joint_mask[..., 6]                                 # (B,T,K)
        rh_ok = joint_mask[..., 7]                                 # (B,T,K)
    
        midhip = 0.5 * (xy[..., 6, :] + xy[..., 7, :])             # (B,T,K,2)
    
        # centroid over joints (robust fallback)
        jm_f = joint_mask.unsqueeze(-1).float()                    # (B,T,K,V,1)
        sum_xy = (xy * jm_f).sum(dim=-2)                           # (B,T,K,2)
        denom = joint_mask.sum(dim=-1).clamp_min(1).unsqueeze(-1).float()  # (B,T,K,1)
        centroid = sum_xy / denom                                  # (B,T,K,2)
    
        both_hips = lh_ok & rh_ok
        only_lh   = lh_ok & ~rh_ok
        only_rh   = rh_ok & ~lh_ok
    
        anchor_xy = torch.where(
            both_hips.unsqueeze(-1),
            midhip,
            torch.where(
                only_lh.unsqueeze(-1),
                xy[..., 6, :],
                torch.where(
                    only_rh.unsqueeze(-1),
                    xy[..., 7, :],
                    centroid,
                ),
            ),
        )                                                          # (B,T,K,2)
    
        anchor_valid = (lh_ok | rh_ok | joint_mask.any(dim=-1))     # (B,T,K) bool'''
    
        # --- build feature vector (NOTE: must match your in_dim in __init__) ---
        feat = []
    
        # bones only (flatten Ebones*2)
        feat.append(bones.reshape(*bones.shape[:-2], -1))           # (B,T,K,2*Ebones)
    
        # anchor
        feat.append(anchor_xy)                                      # (B,T,K,2)
    
        if self.add_masks:
            feat.append(bone_mask.float())                          # (B,T,K,Ebones)
            feat.append(person_mask.float().unsqueeze(-1))          # (B,T,K,1)
            feat.append(anchor_valid.float().unsqueeze(-1))         # (B,T,K,1)
    
        if self.add_bbox:
            '''if bbox_cxcywh is None:
                raise ValueError("add_bbox=True but bbox_cxcywh is None")
            feat.append(bbox_cxcywh)                                # (B,T,K,4)'''
            feat.append(pers_bboxes)                                # (B,T,K,4)
        x = torch.cat(feat, dim=-1)                                 # (B,T,K,in_dim)
        emb = self.mlp(x) # (B,T,K,out_dim)
        
        return emb, person_mask


In [22]:

def masked_mean(x, mask, dim):
    # x: (..., K, E), mask: (..., K) in {0,1}
    w = mask.float().unsqueeze(-1)                # (..., K, 1)
    s = (x * w).sum(dim=dim)                      # (..., E)
    d = w.sum(dim=dim).clamp_min(1.0)             # (..., 1)
    return s / d

def masked_std(x, mask, dim):
    mu = masked_mean(x, mask, dim=dim)            # (..., E)
    w = mask.float().unsqueeze(-1)
    var = (w * (x - mu.unsqueeze(dim))**2).sum(dim=dim) / w.sum(dim=dim).clamp_min(1.0)
    return torch.sqrt(var + 1e-6)

def masked_max(x, mask, dim):
    # set masked-out to very negative so they never win max
    #neg_inf = torch.finfo(x.dtype).min
    neg = -1e9
    x2 = x.masked_fill(~mask.bool().unsqueeze(-1), neg)
    mx = x2.max(dim=dim).values  # (..., E)

    # if all invalid, set output to 0
    all_invalid = (mask.sum(dim=dim) == 0)  # (...) boolean
    if all_invalid.any():
        mx = mx.masked_fill(all_invalid.unsqueeze(-1), 0.0)
    return mx

class PersonPooling(nn.Module):
    """
    emb: (B,T,K,E)
    mask: (B,T,K) boolean, True for valid persons
    """
    def __init__(self, mode="mean_max_std"):
        super().__init__()
        self.mode = mode

    def forward(self, emb, mask=None):
        B, T, K, E = emb.shape
        if mask is None:
            # assume all K are valid
            mask = torch.ones((B, T, K), device=emb.device, dtype=torch.bool)

        if self.mode == "mean":
            return masked_mean(emb, mask, dim=2)                         # (B,T,E)
        if self.mode == "max":
            return masked_max(emb, mask, dim=2)                          # (B,T,E)
        if self.mode == "mean_max":
            mu = masked_mean(emb, mask, dim=2)
            mx = masked_max(emb, mask, dim=2)
            return torch.cat([mu, mx], dim=-1)                           # (B,T,2E)
        if self.mode == "mean_max_std":
            mu = masked_mean(emb, mask, dim=2)
            mx = masked_max(emb, mask, dim=2)
            sd = masked_std(emb, mask, dim=2)
            return torch.cat([mu, mx, sd], dim=-1)                       # (B,T,3E)

        raise ValueError(self.mode)


In [23]:
class TemporalConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, dilation=1):
        super().__init__()
        padding = (kernel_size - 1) * dilation // 2

        self.conv = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            padding=padding,
            dilation=dilation,
        )
        self.bn = nn.BatchNorm1d(out_channels)

    def forward(self, x):
        # x: (B, C_in, T)
        out = self.conv(x)
        out = self.bn(out)
        out = F.relu(out)
        return out

class EventTCNSimple(nn.Module):
    def __init__(self, input_dim, num_classes=1,
                 hidden_dim=64, num_layers=3):
        super().__init__()

        layers = []
        in_ch = input_dim
        dilation = 1
        for _ in range(num_layers):
            layers.append(
                TemporalConvBlock(
                    in_channels=in_ch,
                    out_channels=hidden_dim,
                    kernel_size=3,
                    dilation=dilation,
                )
            )
            in_ch = hidden_dim
            dilation *= 2  # 1,2,4,...

        self.tcn = nn.Sequential(*layers)
        #self.head = nn.Conv1d(hidden_dim, num_classes, kernel_size=1)
        self.fc = nn.Linear(hidden_dim, 1)   # <--- ONE logit

    def forward(self, x):
        # x: (B, T, C)
        x = x.transpose(1, 2)          # (B, C, T)
        feat = self.tcn(x)             # (B, hidden_dim, T)
        #logits = self.head(feat)       # (B, num_classes, T)
        #logits = logits.transpose(1, 2)  # (B, T, num_classes)
        mid = feat.size(-1) // 2
        last_feat = feat[:, :, mid]         # (B, hidden_dim)
        logits = self.fc(last_feat).squeeze(-1)  # (B,)  raw score
        return logits


In [24]:
class EventTCN(nn.Module):
    def __init__(self, input_dim=96, num_classes=1,
                 hidden_dim=64, num_layers=3):
        super().__init__()

        layers = []
        in_ch = input_dim +1
        dilation = 1
        for _ in range(num_layers):
            layers.append(
                TemporalConvBlock(
                    in_channels=in_ch,
                    out_channels=hidden_dim,
                    kernel_size=3,
                    dilation=dilation,
                )
            )
            in_ch = hidden_dim
            dilation *= 2  # 1,2,4,...
        self.mlp = BoneMLPEncoder()
        self.pool = PersonPooling()
        self.tcn = nn.Sequential(*layers)
        #self.head = nn.Conv1d(hidden_dim, num_classes, kernel_size=1)
        self.fc = nn.Linear(hidden_dim, 1)   # <--- ONE logit

    def forward(self, x):
        # x: (B, T, C)
        # pose_xyc: (B,T,K,V,3) or (B,T,K,V,2)
        emb, person_mask = self.mlp(x) # B,T,K,out_dim
        
        scene = self.pool(emb, mask=person_mask)              # (B,T,C)
        
        count = person_mask.sum(dim=2).float()                # (B,T)
        count_norm = count / person_mask.size(2)              # (B,T) for K fixed
        #print(count_norm.unsqueeze(-1).shape)
        scene_count = torch.cat([scene, count_norm.unsqueeze(-1)], dim=-1)  # (B,T,C+1)
        #print(scene_count.shape)
        #B, T, K, E = x.shape

        '''scene_per = (
            scene.permute(0, 2, 1)      # (B, K, E, T)
             .contiguous()
             #.view(B * K, E, T)        # (B*K, E, T)
        )'''
        tcn_in = scene_count.transpose(1,2).contiguous()
        feat = self.tcn(tcn_in)             # (B, hidden_dim, T)
        #logits = self.head(feat)       # (B, num_classes, T)
        #logits = logits.transpose(1, 2)  # (B, T, num_classes)
        #last_feat = feat[:, :, -1]         # (B, hidden_dim)
        mid = feat.size(-1) // 2
        mid_feat = feat[:, :, mid]              # (B, hidden)
        logits = self.fc(mid_feat).squeeze(-1)  # (B,)
        #logits = self.fc(feat)#.squeeze(-1)  # (B,)  raw score
        return logits


In [25]:
from sklearn.model_selection import train_test_split

train_paths, val_paths = train_test_split(json_paths, test_size=0.2, random_state=42)
print(len(train_paths), len(val_paths))
train_dataset = EventJsonDataset(train_paths, K,
                                 window_size=WINDOW_SIZE,
                                 window_step=WINDOW_STEP)

val_dataset = EventJsonDataset(val_paths, K,
                               window_size=WINDOW_SIZE,
                               window_step=WINDOW_STEP)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
batches_with_y_not_0_or_1 = 0

for batch_features, batch_labels in train_loader:
    # Check if any label in the current batch is neither 0 nor 1
    # torch.logical_and checks element-wise for both conditions
    # .any() checks if at least one element in the resulting boolean tensor is True
    if torch.logical_and(batch_labels != 0, batch_labels != 1).any():
        batches_with_y_not_0_or_1 += 1

print(f"Number of batches with labels not 0 or 1: {batches_with_y_not_0_or_1}")
batches_with_y_not_0_or_1 = 0

for batch_features, batch_labels in val_loader:
    # Check if any label in the current batch is neither 0 nor 1
    # torch.logical_and checks element-wise for both conditions
    # .any() checks if at least one element in the resulting boolean tensor is True
    if torch.logical_and(batch_labels != 0, batch_labels != 1).any():
        batches_with_y_not_0_or_1 += 1

print(f"Number of batches with labels not 0 or 1: {batches_with_y_not_0_or_1}")

69 18
Total windows: 16080
Total windows: 2171
Number of batches with labels not 0 or 1: 0
Number of batches with labels not 0 or 1: 0


In [27]:
import numpy as np

all_y = []
for _, y in train_dataset:
    all_y.append(y)  # y is 0 or 1
all_y = np.array(all_y)
n_neg = (all_y == 0).sum()
n_pos = (all_y > 0).sum()
print("n_neg, n_pos =", n_neg, n_pos)
# Create a boolean mask for elements > 0
condition_gt_0 = all_y > 0

# Create a boolean mask for elements < 1
condition_lt_1 = all_y < 1

# Combine the conditions using the logical AND operator (&)
combined_condition = condition_gt_0 & condition_lt_1
all_y[combined_condition]

n_neg, n_pos = 13572 2684


array([0.375, 0.875, 0.75 , 0.25 , 0.125, 0.625, 0.875, 0.375, 0.75 ,
       0.25 , 0.125, 0.625, 0.75 , 0.5  , 0.125, 0.625, 0.75 , 0.5  ,
       0.625, 0.125, 0.125, 0.625, 0.875, 0.375, 0.5  , 0.625, 0.125,
       0.125, 0.625, 0.625, 0.125, 0.75 , 0.75 , 0.25 , 0.25 , 0.75 ,
       0.75 , 0.25 , 0.75 , 0.625, 0.125, 0.25 , 0.75 , 0.875, 0.75 ,
       0.25 , 0.375, 0.875, 0.625, 0.125, 0.125, 0.625, 0.875, 0.375,
       0.25 , 0.75 , 0.875, 0.375, 0.5  , 0.5  , 0.125, 0.625, 0.75 ,
       0.75 , 0.625, 0.125, 0.5  , 0.75 , 0.25 , 0.5  , 0.5  , 0.125,
       0.625, 0.875, 0.375, 0.5  , 0.75 , 0.75 , 0.25 , 0.375, 0.875,
       0.875, 0.375, 0.5  , 0.75 , 0.25 , 0.25 , 0.75 , 0.875, 0.875,
       0.375, 0.25 , 0.75 , 0.875, 0.875, 0.75 , 0.25 , 0.25 , 0.75 ,
       0.875, 0.375, 0.125, 0.625, 0.625, 0.125, 0.25 , 0.75 , 0.125,
       0.625, 0.5  , 0.25 , 0.75 , 0.875, 0.875, 0.375, 0.25 , 0.75 ,
       0.875, 0.375, 0.5  , 0.75 , 0.25 , 0.25 , 0.75 , 0.75 , 0.25 ,
       0.125, 0.625,

In [26]:
import numpy as np
from sklearn.metrics import precision_recall_curve, average_precision_score

def prf_at_threshold(y_true, probs, thr):
    """
    y_true: (N,) int {0,1}
    probs:  (N,) float in [0,1]
    """
    pred = (probs >= thr).astype(np.int32)
    tp = np.sum((pred == 1) & (y_true == 1))
    fp = np.sum((pred == 1) & (y_true == 0))
    fn = np.sum((pred == 0) & (y_true == 1))
    precision = tp / (tp + fp + 1e-9)
    recall    = tp / (tp + fn + 1e-9)
    f1        = 2 * precision * recall / (precision + recall + 1e-9)
    return precision, recall, f1

def best_f1_threshold(y_true, probs):
    """
    Returns: best_thr, best_p, best_r, best_f1
    Uses sklearn precision_recall_curve thresholds.
    """
    precision, recall, thresholds = precision_recall_curve(y_true, probs)
    # precision and recall have length len(thresholds)+1
    # align by dropping last point
    precision = precision[:-1]
    recall = recall[:-1]
    f1 = 2 * precision * recall / (precision + recall + 1e-9)
    i = int(np.nanargmax(f1))
    return float(thresholds[i]), float(precision[i]), float(recall[i]), float(f1[i])

def threshold_for_recall(y_true, probs, recall_target=0.7):
    """
    Choose the highest threshold that still achieves recall >= recall_target.
    Higher threshold => higher precision typically, so we choose highest that meets recall.
    Returns: thr, p, r, f1
    """
    precision, recall, thresholds = precision_recall_curve(y_true, probs)
    precision = precision[:-1]
    recall = recall[:-1]
    thresholds = thresholds

    ok = np.where(recall >= recall_target)[0]
    if len(ok) == 0:
        # cannot reach target recall; return lowest threshold (most permissive)
        thr = float(thresholds[0])
        p, r, f1 = prf_at_threshold(y_true, probs, thr)
        return thr, p, r, f1

    i = ok[-1]  # highest threshold among those meeting recall
    thr = float(thresholds[i])
    return thr, float(precision[i]), float(recall[i]), float(
        2 * precision[i] * recall[i] / (precision[i] + recall[i] + 1e-9)
    )

def threshold_for_precision(y_true, probs, precision_target=0.5):
    """
    Choose the lowest threshold that achieves precision >= target.
    Lower threshold => higher recall typically, so choose lowest that meets precision.
    Returns: thr, p, r, f1
    """
    precision, recall, thresholds = precision_recall_curve(y_true, probs)
    precision = precision[:-1]
    recall = recall[:-1]

    ok = np.where(precision >= precision_target)[0]
    if len(ok) == 0:
        thr = float(thresholds[-1])  # strictest
        p, r, f1 = prf_at_threshold(y_true, probs, thr)
        return thr, p, r, f1

    i = ok[0]  # lowest threshold that meets precision
    thr = float(thresholds[i])
    return thr, float(precision[i]), float(recall[i]), float(
        2 * precision[i] * recall[i] / (precision[i] + recall[i] + 1e-9)
    )


In [27]:
DEVICE  = "cpu"
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, precision_recall_curve
import copy
model = EventTCN().to(DEVICE)
#criterion = nn.CrossEntropyLoss()
criterion = nn.BCEWithLogitsLoss()#pos_weight=torch.tensor([4.0], device=DEVICE))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 15
PATIENCE = 4
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

def batch_event_accuracy(probs, y, thr=0.5):
    # probs, y: (B,) floats in [0,1]
    pred_bin = (probs >= thr)
    y_bin = (y >= thr)
    return (pred_bin == y_bin).float().mean().item()
    
best_val_loss = float("inf")
best_val_acc = 0.0
best_auprc = -1.0
best_model_state = None
best_model_state_acc = None
epochs_no_improve = 0
best_epoch = 0

for epoch in range(EPOCHS):
    # --- train ---
    model.train()
    running_loss = 0.0
    running_acc = 0
    total = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(DEVICE)   # (B, T, C)
        #y_batch = y_batch.float().to(DEVICE)# (B, )
        y = y_batch.to(DEVICE).float().view(-1)

        logits = model(X_batch).view(-1)        # (B, 1)
        base = F.binary_cross_entropy_with_logits(logits, y, reduction="none")
        #loss = criterion(logits, y)
        #B, T, Cc = logits.shape

        #loss = criterion(
         #   logits.reshape(B*T, Cc),
         #   y_batch.view(-1)
        #)
        alpha = 3.0
        w = 1.0 + alpha * y                  # y=0 -> 1.0, y=1 -> 4.0
        loss = (base * w).mean()
        B = y.size(0)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * B# * T

        probs = torch.sigmoid(logits)#.argmax(dim=-1)          # (B, T)
        #correct += (preds == y_batch).sum().item()
        running_acc += batch_event_accuracy(probs, y) * B
        total   += B# * T

    train_loss = running_loss / total
    train_acc  = running_acc / total

    # --- validation ---
    model.eval()
    val_running_loss = 0.0
    val_running_acc = 0
    val_total = 0
    tp = fp = fn = 0
    all_probs = []
    all_ytrue = []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(DEVICE)
            y = y_batch.to(DEVICE).float().view(-1)      # (B,)
            #y_batch = y_batch.to(DEVICE)
            #print(y_batch.shape)
            logits = model(X_batch).view(-1)
            #logits =
            #Cc = logits.shape
            #logits = logits.view(-1)                # (B,)
            #y = y_batch.view(-1).float() 
            B = y.size(0)
            base = F.binary_cross_entropy_with_logits(logits, y)

            val_running_loss += base.item() * B

            
            probs = torch.sigmoid(logits)           # <-- this is the function you forgot
            
            y_bin = (y >= 0.5)
            pred_bin = (probs >= 0.5)
            
            tp += (pred_bin & y_bin).sum().item()
            fp += (pred_bin & ~y_bin).sum().item()
            fn += (~pred_bin & y_bin).sum().item()
            all_probs.append(probs.detach().cpu())
            all_ytrue.append(y_bin.detach().cpu())
        
            #preds = (probs >= 0.5).float()  
            val_running_acc += batch_event_accuracy(probs, y) * B
            val_total += B
        val_loss = val_running_loss / val_total
        val_acc  = val_running_acc / val_total
            #preds = logits
            #val_correct += (preds == y_batch).sum().item()
    precision = tp / (tp + fp + 1e-9)
    recall    = tp / (tp + fn + 1e-9)
    f1        = 2*precision*recall / (precision + recall + 1e-9)
    #val_total   += y.size(0)
    all_probs = torch.cat(all_probs).numpy()
    all_ytrue = torch.cat(all_ytrue).numpy().astype(int)
    auprc = average_precision_score(all_ytrue, all_probs)  # AP = AUPRC
    
    #print("P/R/F1:", precision, recall, f1, "AUPRC:", auprc)
    
    # (A) Best F1 threshold
    thr_f1, p_f1, r_f1, f1_best = best_f1_threshold(all_ytrue, all_probs)
    
    # (B) Example: threshold for recall >= 0.70
    thr_r, p_r, r_r, f1_r = threshold_for_recall(all_ytrue, all_probs, recall_target=0.70)
    
    # (C) Example: threshold for precision >= 0.50
    thr_p, p_p, r_p, f1_p = threshold_for_precision(all_ytrue, all_probs, precision_target=0.50)
    
    print(f"AUPRC: {auprc:.4f}")
    print(f"Best-F1: thr={thr_f1:.3f} P={p_f1:.3f} R={r_f1:.3f} F1={f1_best:.3f}")
    print(f"Recall>=0.70: thr={thr_r:.3f} P={p_r:.3f} R={r_r:.3f} F1={f1_r:.3f}")
    print(f"Prec>=0.50: thr={thr_p:.3f} P={p_p:.3f} R={r_p:.3f} F1={f1_p:.3f}")

    
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch+1:02d} | "
          f"train_loss={train_loss:.4f}, train_acc={train_acc:.3f} | "
          f"val_loss={val_loss:.4f}, val_acc={val_acc:.3f}")
    '''if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(model.state_dict())
        best_epoch = epoch
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1'''
    
    if auprc > best_auprc:
        best_auprc = auprc
        best_model_state_auprc = copy.deepcopy(model.state_dict())
        best_epoch = epoch
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        
    '''if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state_acc = copy.deepcopy(model.state_dict())'''
        
    # --- early stopping (optional) ---
    if PATIENCE is not None and epochs_no_improve >= PATIENCE:
        print(f"Early stopping at epoch {epoch+1}, "
              f"no improvement for {PATIENCE} epochs. Best model by AUPRC saved after {best_epoch+1} epoch/s.")
        break

PATH = "tcn_best_by_auprc.pt"  # Choose a suitable file path and name
torch.save(best_model_state_auprc, PATH)
#torch.save(best_model_state_acc,  "tcn_best_by_acc.pth")

AUPRC: 0.5136
Best-F1: thr=0.406 P=0.407 R=0.655 F1=0.502
Recall>=0.70: thr=0.225 P=0.372 R=0.700 F1=0.486
Prec>=0.50: thr=0.889 P=0.500 R=0.424 F1=0.459
Epoch 01 | train_loss=0.4834, train_acc=0.871 | val_loss=0.8240, val_acc=0.682
AUPRC: 0.4749
Best-F1: thr=0.851 P=0.389 R=0.651 F1=0.487
Recall>=0.70: thr=0.765 P=0.372 R=0.700 F1=0.485
Prec>=0.50: thr=0.975 P=0.500 R=0.384 F1=0.434
Epoch 02 | train_loss=0.3730, train_acc=0.898 | val_loss=1.2569, val_acc=0.557
AUPRC: 0.5272
Best-F1: thr=0.158 P=0.368 R=0.818 F1=0.508
Recall>=0.70: thr=0.381 P=0.382 R=0.700 F1=0.494
Prec>=0.50: thr=0.944 P=0.500 R=0.429 F1=0.462
Epoch 03 | train_loss=0.3054, train_acc=0.923 | val_loss=0.9992, val_acc=0.661
AUPRC: 0.3922
Best-F1: thr=0.215 P=0.410 R=0.693 F1=0.515
Recall>=0.70: thr=0.194 P=0.404 R=0.700 F1=0.512
Prec>=0.50: thr=0.999 P=0.500 R=0.060 F1=0.107
Epoch 04 | train_loss=0.2475, train_acc=0.937 | val_loss=1.1524, val_acc=0.699
AUPRC: 0.3861
Best-F1: thr=0.080 P=0.415 R=0.751 F1=0.534
Recall>=0.

In [30]:
DEVICE  = "cpu"
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, precision_recall_curve
import copy
model = EventTCNSimple(input_dim=NUM_FEATURES*K + 1).to(DEVICE)
#criterion = nn.CrossEntropyLoss()
criterion = nn.BCEWithLogitsLoss()#pos_weight=torch.tensor([4.0], device=DEVICE))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 15
PATIENCE = 4
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

def batch_event_accuracy(probs, y, thr=0.5):
    # probs, y: (B,) floats in [0,1]
    pred_bin = (probs >= thr)
    y_bin = (y >= thr)
    return (pred_bin == y_bin).float().mean().item()
    
best_val_loss = float("inf")
best_val_acc = 0.0
best_auprc = -1.0
best_model_state = None
best_model_state_acc = None
epochs_no_improve = 0
best_epoch = 0

for epoch in range(EPOCHS):
    # --- train ---
    model.train()
    running_loss = 0.0
    running_acc = 0
    total = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(DEVICE)   # (B, T, C)
        #y_batch = y_batch.float().to(DEVICE)# (B, )
        y = y_batch.to(DEVICE).float().view(-1)

        logits = model(X_batch).view(-1)        # (B, 1)
        base = F.binary_cross_entropy_with_logits(logits, y, reduction="none")
        #loss = criterion(logits, y)
        #B, T, Cc = logits.shape

        #loss = criterion(
         #   logits.reshape(B*T, Cc),
         #   y_batch.view(-1)
        #)
        alpha = 3.0
        w = 1.0 + alpha * y                  # y=0 -> 1.0, y=1 -> 4.0
        loss = (base * w).mean()
        B = y.size(0)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * B# * T

        probs = torch.sigmoid(logits)#.argmax(dim=-1)          # (B, T)
        #correct += (preds == y_batch).sum().item()
        running_acc += batch_event_accuracy(probs, y) * B
        total   += B# * T

    train_loss = running_loss / total
    train_acc  = running_acc / total

    # --- validation ---
    model.eval()
    val_running_loss = 0.0
    val_running_acc = 0
    val_total = 0
    tp = fp = fn = 0
    all_probs = []
    all_ytrue = []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(DEVICE)
            y = y_batch.to(DEVICE).float().view(-1)      # (B,)
            #y_batch = y_batch.to(DEVICE)
            #print(y_batch.shape)
            logits = model(X_batch).view(-1)
            #logits =
            #Cc = logits.shape
            #logits = logits.view(-1)                # (B,)
            #y = y_batch.view(-1).float() 
            B = y.size(0)
            base = F.binary_cross_entropy_with_logits(logits, y)

            val_running_loss += base.item() * B

            
            probs = torch.sigmoid(logits)           # <-- this is the function you forgot
            
            y_bin = (y >= 0.5)
            pred_bin = (probs >= 0.5)
            
            tp += (pred_bin & y_bin).sum().item()
            fp += (pred_bin & ~y_bin).sum().item()
            fn += (~pred_bin & y_bin).sum().item()
            all_probs.append(probs.detach().cpu())
            all_ytrue.append(y_bin.detach().cpu())
        
            #preds = (probs >= 0.5).float()  
            val_running_acc += batch_event_accuracy(probs, y) * B
            val_total += B
        val_loss = val_running_loss / val_total
        val_acc  = val_running_acc / val_total
            #preds = logits
            #val_correct += (preds == y_batch).sum().item()
    precision = tp / (tp + fp + 1e-9)
    recall    = tp / (tp + fn + 1e-9)
    f1        = 2*precision*recall / (precision + recall + 1e-9)
    #val_total   += y.size(0)
    all_probs = torch.cat(all_probs).numpy()
    all_ytrue = torch.cat(all_ytrue).numpy().astype(int)
    auprc = average_precision_score(all_ytrue, all_probs)  # AP = AUPRC
    
    #print("P/R/F1:", precision, recall, f1, "AUPRC:", auprc)
    
    # (A) Best F1 threshold
    thr_f1, p_f1, r_f1, f1_best = best_f1_threshold(all_ytrue, all_probs)
    
    # (B) Example: threshold for recall >= 0.70
    thr_r, p_r, r_r, f1_r = threshold_for_recall(all_ytrue, all_probs, recall_target=0.70)
    
    # (C) Example: threshold for precision >= 0.50
    thr_p, p_p, r_p, f1_p = threshold_for_precision(all_ytrue, all_probs, precision_target=0.50)
    
    print(f"AUPRC: {auprc:.4f}")
    print(f"Best-F1: thr={thr_f1:.3f} P={p_f1:.3f} R={r_f1:.3f} F1={f1_best:.3f}")
    print(f"Recall>=0.70: thr={thr_r:.3f} P={p_r:.3f} R={r_r:.3f} F1={f1_r:.3f}")
    print(f"Prec>=0.50: thr={thr_p:.3f} P={p_p:.3f} R={r_p:.3f} F1={f1_p:.3f}")

    
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch+1:02d} | "
          f"train_loss={train_loss:.4f}, train_acc={train_acc:.3f} | "
          f"val_loss={val_loss:.4f}, val_acc={val_acc:.3f}")
    '''if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(model.state_dict())
        best_epoch = epoch
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1'''
    
    if auprc > best_auprc:
        best_auprc = auprc
        best_model_state_auprc = copy.deepcopy(model.state_dict())
        best_epoch = epoch
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        
    '''if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state_acc = copy.deepcopy(model.state_dict())'''
        
    # --- early stopping (optional) ---
    if PATIENCE is not None and epochs_no_improve >= PATIENCE:
        print(f"Early stopping at epoch {epoch+1}, "
              f"no improvement for {PATIENCE} epochs. Best model by AUPRC saved after {best_epoch+1} epoch/s.")
        break

PATH = "tcn_best_by_auprc.pt"  # Choose a suitable file path and name
torch.save(best_model_state_auprc, PATH)
#torch.save(best_model_state_acc,  "tcn_best_by_acc.pth")

AUPRC: 0.3399
Best-F1: thr=0.032 P=0.281 R=0.913 F1=0.429
Recall>=0.70: thr=0.452 P=0.292 R=0.700 F1=0.412
Prec>=0.50: thr=1.000 P=0.500 R=0.129 F1=0.205
Epoch 01 | train_loss=0.4627, train_acc=0.871 | val_loss=2.1632, val_acc=0.499
AUPRC: 0.2330
Best-F1: thr=0.009 P=0.277 R=0.838 F1=0.416
Recall>=0.70: thr=0.056 P=0.280 R=0.700 F1=0.400
Prec>=0.50: thr=1.000 P=0.000 R=0.000 F1=0.000
Epoch 02 | train_loss=0.2658, train_acc=0.933 | val_loss=1.6806, val_acc=0.546
AUPRC: 0.3077
Best-F1: thr=0.003 P=0.269 R=0.880 F1=0.412
Recall>=0.70: thr=0.099 P=0.279 R=0.700 F1=0.399
Prec>=0.50: thr=1.000 P=0.500 R=0.015 F1=0.028
Epoch 03 | train_loss=0.1405, train_acc=0.967 | val_loss=2.4163, val_acc=0.536
AUPRC: 0.2499
Best-F1: thr=0.001 P=0.280 R=0.807 F1=0.416
Recall>=0.70: thr=0.006 P=0.277 R=0.700 F1=0.397
Prec>=0.50: thr=1.000 P=0.000 R=0.000 F1=0.000
Epoch 04 | train_loss=0.0772, train_acc=0.983 | val_loss=2.8581, val_acc=0.532
AUPRC: 0.3202
Best-F1: thr=0.002 P=0.286 R=0.865 F1=0.429
Recall>=0.

In [28]:
import copy

model = EventTCNSimple(input_dim=NUM_FEATURES*K + 1, num_classes=NUM_CLASSES).to(DEVICE)
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(4.0, device=DEVICE))  # pos_weight is a scalar tensor
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 15
PATIENCE = 8
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

best_val_loss = float("inf")
best_val_acc = 0.0

best_model_state = None
best_model_state_acc = None
epochs_no_improve = 0
best_epoch = 0

for epoch in range(EPOCHS):
    # --- train ---
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for X_batch, y_batch in train_loader:
        # X_batch: (B, T, C)
        # y_batch: (B,)  in [0,1] (ramp) or {0,1}
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE).float()

        logits = model(X_batch)          # (B,) raw scores
        loss = criterion(logits, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        B = y_batch.size(0)
        running_loss += loss.item() * B

        # accuracy: binarize both labels and predictions
        with torch.no_grad():
            probs = torch.sigmoid(logits)                # (B,)
            preds = (probs >= 0.5).float()               # 0 or 1
            labels_bin = (y_batch >= 0.5).float()        # for ramp labels

            correct += (preds == labels_bin).sum().item()
            total   += B

    train_loss = running_loss / total
    train_acc  = correct / total

    # --- validation ---
    model.eval()
    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE).float()

            logits = model(X_batch)       # (B,)
            loss = criterion(logits, y_batch)

            B = y_batch.size(0)
            val_running_loss += loss.item() * B

            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).float()
            labels_bin = (y_batch >= 0.5).float()

            val_correct += (preds == labels_bin).sum().item()
            val_total   += B

    val_loss = val_running_loss / val_total
    val_acc  = val_correct / val_total
        
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch+1:02d} | "
          f"train_loss={train_loss:.4f}, train_acc={train_acc:.3f} | "
          f"val_loss={val_loss:.4f}, val_acc={val_acc:.3f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(model.state_dict())
        best_epoch = epoch
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state_acc = copy.deepcopy(model.state_dict())
        
    # --- early stopping (optional) ---
    if PATIENCE is not None and epochs_no_improve >= PATIENCE:
        print(f"Early stopping at epoch {epoch+1}, "
              f"no improvement for {PATIENCE} epochs. Best model by loss saved after {best_epoch} epoch/s.")
        break

PATH = "tcn_best_by_loss.pt"  # Choose a suitable file path and name
torch.save(best_model_state, PATH)
torch.save(best_model_state_acc,  "tcn_best_by_acc.pth")

Epoch 01 | train_loss=0.4694, train_acc=0.867 | val_loss=2.1234, val_acc=0.532
Epoch 02 | train_loss=0.2526, train_acc=0.938 | val_loss=4.1207, val_acc=0.574
Epoch 03 | train_loss=0.1635, train_acc=0.964 | val_loss=3.1212, val_acc=0.554
Epoch 04 | train_loss=0.0749, train_acc=0.983 | val_loss=3.9305, val_acc=0.501
Epoch 05 | train_loss=0.0585, train_acc=0.987 | val_loss=5.2945, val_acc=0.502
Epoch 06 | train_loss=0.0465, train_acc=0.990 | val_loss=5.2534, val_acc=0.534
Epoch 07 | train_loss=0.0498, train_acc=0.989 | val_loss=5.7453, val_acc=0.537
Epoch 08 | train_loss=0.0368, train_acc=0.992 | val_loss=5.8272, val_acc=0.555
Epoch 09 | train_loss=0.0230, train_acc=0.994 | val_loss=6.0671, val_acc=0.530
Early stopping at epoch 9, no improvement for 8 epochs. Best model by loss saved after 0 epoch/s.


In [ ]:
import numpy as np

def evaluate_per_class(model, loader, num_classes=3):
    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE).float()

            logits = model(X_batch)
            probs = torch.sigmoid(logits)
            #preds = logits.argmax(dim=-1)  # (B, T)

            all_preds.append(preds.cpu().numpy())
            all_targets.append(y_batch.cpu().numpy())

    y_true = np.concatenate(all_targets)
    y_pred = np.concatenate(all_preds)

    for c, name in zip(range(2), ["none", "tension"]):
        mask = (y_true == c)
        if mask.sum() == 0:
            print(f"class {c} ({name}): no true examples")
            continue

        tp = ((y_true == c) & (y_pred == c)).sum()
        fn = ((y_true == c) & (y_pred != c)).sum()
        fp = ((y_true != c) & (y_pred == c)).sum()

        recall = tp / (tp + fn + 1e-8)
        precision = tp / (tp + fp + 1e-8)

        print(f"class {c} ({name}): "
              f"precision={precision:.3f}, recall={recall:.3f}, support={mask.sum()}")

evaluate_per_class(model, val_loader, num_classes=2)


In [33]:
import numpy as np
import torch

def evaluate_per_class(model_path, loader, threshold=0.5):
    """
    Binary evaluation: class 0 = none, class 1 = tension/event
    threshold: decision threshold on sigmoid(prob) for event.
    """
    # Load the state dictionary
    state_dict = torch.load(model_path, weights_only=True)
    model = EventTCN()
    # Load the state dictionary into the model
    model.load_state_dict(state_dict)
    model.eval()
    all_scores = []   # sigmoid outputs
    all_targets = []  # true labels (possibly soft) from dataset

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE).float()   # (B,)

            logits = model(X_batch)                # (B,)
            probs = torch.sigmoid(logits)          # (B,)

            all_scores.append(probs.cpu().numpy())
            all_targets.append(y_batch.cpu().numpy())

    y_true = np.concatenate(all_targets)           # soft labels in [0,1]
    y_score = np.concatenate(all_scores)           # model scores in [0,1]

    # Binarize both: decide what is "event" vs "none"
    y_true_bin = (y_true >= threshold).astype(int)       # ground truth 0/1
    y_pred_bin = (y_score >= threshold).astype(int)  # predictions 0/1

    for c, name in zip([0, 1], ["none", "tension"]):
        mask = (y_true_bin == c)
        support = mask.sum()
        if support == 0:
            print(f"class {c} ({name}): no true examples")
            continue

        tp = ((y_true_bin == c) & (y_pred_bin == c)).sum()
        fn = ((y_true_bin == c) & (y_pred_bin != c)).sum()
        fp = ((y_true_bin != c) & (y_pred_bin == c)).sum()

        recall = tp / (tp + fn + 1e-8)
        precision = tp / (tp + fp + 1e-8)

        print(
            f"class {c} ({name}): "
            f"precision={precision:.3f}, recall={recall:.3f}, support={support}"
        )

# usage
#evaluate_per_class(model, val_loader, threshold=0.5)
for th in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    print(f"\nThreshold = {th}")
    evaluate_per_class('tcn_best_by_acc.pth', val_loader, threshold=th)
for th in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    print(f"\nThreshold = {th}")
    evaluate_per_class('tcn_best_by_loss.pt', val_loader, threshold=th)


Threshold = 0.2


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cpu and cuda:0! (when checking argument for argument mat1 in method wrapper_CUDA_addmm)

In [25]:
import numpy as np
import torch

def evaluate_per_class(model_path, loader, y_thr=0.5, pred_thr=0.5):
    state_dict = torch.load(model_path, weights_only=True)
    model = EventTCN().to(DEVICE)
    model.load_state_dict(state_dict)
    model.eval()

    all_scores = []
    all_targets = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE).float().view(-1)

            logits = model(X_batch).view(-1)
            probs = torch.sigmoid(logits)

            all_scores.append(probs.detach().cpu().numpy())
            all_targets.append(y_batch.detach().cpu().numpy())

    y_true = np.concatenate(all_targets)   # soft labels in [0,1]
    y_score = np.concatenate(all_scores)   # probs in [0,1]

    # FIXED ground truth definition
    y_true_bin = (y_true >= y_thr).astype(np.int32)

    # SWEPT prediction threshold
    y_pred_bin = (y_score >= pred_thr).astype(np.int32)

    for c, name in [(0, "none"), (1, "tension")]:
        mask = (y_true_bin == c)
        support = int(mask.sum())
        if support == 0:
            print(f"class {c} ({name}): no true examples")
            continue

        tp = int(((y_true_bin == c) & (y_pred_bin == c)).sum())
        fn = int(((y_true_bin == c) & (y_pred_bin != c)).sum())
        fp = int(((y_true_bin != c) & (y_pred_bin == c)).sum())

        recall = tp / (tp + fn + 1e-8)
        precision = tp / (tp + fp + 1e-8)

        print(f"class {c} ({name}): precision={precision:.3f}, recall={recall:.3f}, support={support}")


In [26]:
for th in [0.2,0.3,0.4,0.5,0.6,0.7,0.8]:
    print(f"\nPred threshold = {th}")
    evaluate_per_class("tcn_best_by_acc.pth", val_loader, y_thr=0.5, pred_thr=th)


Pred threshold = 0.2


/home/anna/.local/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


class 0 (none): precision=0.847, recall=0.745, support=1621
class 1 (tension): precision=0.445, recall=0.602, support=550

Pred threshold = 0.3
class 0 (none): precision=0.842, recall=0.771, support=1621
class 1 (tension): precision=0.460, recall=0.575, support=550

Pred threshold = 0.4
class 0 (none): precision=0.830, recall=0.783, support=1621
class 1 (tension): precision=0.452, recall=0.525, support=550

Pred threshold = 0.5
class 0 (none): precision=0.818, recall=0.797, support=1621
class 1 (tension): precision=0.443, recall=0.476, support=550

Pred threshold = 0.6
class 0 (none): precision=0.803, recall=0.807, support=1621
class 1 (tension): precision=0.423, recall=0.416, support=550

Pred threshold = 0.7
class 0 (none): precision=0.793, recall=0.819, support=1621
class 1 (tension): precision=0.410, recall=0.371, support=550

Pred threshold = 0.8
class 0 (none): precision=0.773, recall=0.837, support=1621
class 1 (tension): precision=0.364, recall=0.275, support=550


In [24]:
val_loader

In [ ]:
X_win, y_win = val_dataset[0]    # (T, C), (T,)
X_win = X_win.unsqueeze(0).to(DEVICE)  # add batch dim: (1, T, C)
y_win = y_win.numpy()                  # for plotting

model.eval()
with torch.no_grad():
    logits = model(X_win)        # (1, T, num_classes)
    preds = logits.argmax(dim=-1).cpu().numpy().reshape(-1)  # (T,)


In [ ]:
print("True:      ", y_win)
print("Predicted: ", preds)


In [ ]:
import matplotlib.pyplot as plt

T = len(y_win)
t = range(T)

plt.figure(figsize=(10, 3))
plt.step(t, y_win, where="mid", label="true", linewidth=2)
plt.step(t, preds, where="mid", label="pred", linestyle="--")
plt.yticks([0, 1, 2], ["none", "tension", "fight"])
plt.xlabel("time step in window")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import numpy as np

def count_labels(dataset):
    all_y = []
    for _, y_win in dataset:      # y_win: (T,)
        all_y.append(y_win.numpy())
    all_y = np.concatenate(all_y)
    unique, counts = np.unique(all_y, return_counts=True)
    print(dict(zip(unique, counts)))

count_labels(train_dataset)
count_labels(val_dataset)
